In [0]:
# This notebook will:
# Save Gold tables

In [0]:
spark.conf.set(
  "fs.azure.account.key.mystorage.dfs.core.windows.net",
  "ACCESS_KEY_REMOVED"
)

In [0]:
gold_path = "abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/gold/"
agg_path = gold_path + "03_aggregation/"

In [0]:
silver_path = "abfss://finance-data-container@financialpipelineadls2.dfs.core.windows.net/silver/"

df_acct = spark.read.format("delta").load(silver_path + "accounts")
df_cust = spark.read.format("delta").load(silver_path + "customers")
df_loan = spark.read.format("delta").load(silver_path + "loans")
df_tran = spark.read.format("delta").load(silver_path + "transactions")
df_ledg = spark.read.format("delta").load(silver_path + "ledger")

In [0]:
# Report 1 :
# Customers with Balance > $1B
df_r1 = df_acct.filter(df_acct.balance > 1000000000) \
               .select("customer_id", "balance")

df_r1.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_01_high_value_accounts")

In [0]:
# Report 2 :
# 2. Customers (name, email, phone, zip code) from country France.
df_r2 = df_cust.filter(df_cust.country == "France") \
               .select("first_name","last_name","email","phone_number","postal_code")

df_r2.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_02_france_customers")

In [0]:
# Report 3 :
# List of all unique countries.
df_r3 = df_cust.select("country").distinct()

df_r3.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_03_unique_countries")

In [0]:
# Report 4 :
# Transaction details of amount more than $2000.00.
df_r4 = df_tran.filter(df_tran.amount > 2000) \
               .drop("currency_code","tran_status")

df_r4.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_04_high_value_transactions")


In [0]:
# Report 5 : 
# Highest Loan Taken :
from pyspark.sql.functions import max
df_r5 = df_loan.select(
    max("principal_amount").alias("highest_loan_amount")
)

df_r5.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_05_highest_loan")


In [0]:
import pyspark.sql.functions as F

In [0]:
# Report 6 : Top 5 business Loans + Customer info.
# 1. Filter business Loans 
df_business = df_loan.filter(F.col("loan_type") == "Business")

# 2. Join with Customers
df_join = df_business.join(df_cust, "customer_id", "inner")

# 3. Sort & Limit
df_r6 = df_join.orderBy(F.desc("principal_amount")).limit(5)

# Saving to Gold
df_r6.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_06_top5_business_loans")

print("Report 6 Created")

Report 6 Created


In [0]:
# Report 7 
# 10 Customers with Lowest Balances
df_r7 = df_acct.orderBy(F.col("balance").asc()).limit(10)

df_r7.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_07_lowest_10_balances")

print("Report 7 Created")

Report 7 Created


In [0]:
# Report 8
# Interest Rate Analysis 
df_r8 = df_loan.select("interest_rate", "term_month") \
    .describe()

df_r8.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_08_interest_analysis")

print("Report 8 Created")

Report 8 Created


In [0]:
# Report 9
# Highest Ledger Type Entry
# count Ledger Entries and Get Top one.
df_r9 = df_ledg.groupBy("ledger_type") \
    .agg(F.count("*").alias("total_entries")) \
    .orderBy(F.desc("total_entries")) \
    .limit(1)

df_r9.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_09_highest_ledger_type")

print("Report 9 Created")

Report 9 Created


In [0]:
# Report 10 
# Total Pending transactions 
df_r10 = df_tran.filter(F.col("tran_status") == "P") \
    .groupBy("tran_status") \
    .agg(F.count("*").alias("total_pending_transactions"))

df_r10.write.format("delta") \
    .mode("overwrite") \
    .save(agg_path + "report_10_total_pending_transactions")

print("Report 10 Created")

Report 10 Created
